In [ ]:
import pymysql
import pandas as pd

# DB 연결
conn = pymysql.connect(
    host= 'mp.smhrd.or.kr', port= 3306,
    db= 'lang9_core_overfit', user= 'lang9_core_overfit',
    password= 'overfit9',
    charset='utf8mb4'
)
cursor = conn.cursor()

# CSV 로드
df_allergy = pd.read_csv('allergy_check.csv', encoding='utf-8-sig')
df_allergy = df_allergy.dropna()
df_allergy['allergy_check'] = df_allergy['allergy_check'].str.strip()

# CSV 내부 중복 제거
before = len(df_allergy)
df_allergy = df_allergy.drop_duplicates(subset=['allergy_check'])
after = len(df_allergy)
print(f'CSV 내부 중복 제거: {before - after}개 제거 → {after}개')

# DB에 이미 있는 값 가져오기
cursor.execute('SELECT allergy_check FROM t_ingredient')
existing = set(row[0] for row in cursor.fetchall())
print(f'DB 기존 성분 수: {len(existing)}개')

# 중복 제외하고 적재
insert_sql = "INSERT INTO t_ingredient (allergy_check) VALUES (%s)"

batch = []
skip = 0
for _, row in df_allergy.iterrows():
    val = str(row['allergy_check']).strip()
    if val and val not in existing:
        batch.append((val,))
    else:
        skip += 1

if batch:
    cursor.executemany(insert_sql, batch)
    conn.commit()

print(f'✅ 적재 완료 - 신규: {len(batch)}개 / 중복 스킵: {skip}개')

# 검증
cursor.execute('SELECT COUNT(*) FROM t_ingredient')
print(f'DB 전체 성분 수: {cursor.fetchone()[0]}개')

cursor.close()
conn.close()
print('✅ DB 연결 종료')